In [ ]:
import h5py
import numpy as np
import os

# =============================================================
# COMP8851 — RONIN Preprocessing Script
# Group: MDS-IoT-Yu2 | Supervisor: Dr Yu Zhang
# 
# Run this script ONCE after downloading the full RONIN dataset.
# It will save the preprocessed arrays to disk as .npy files.
# After running, use the loading snippet in your model notebook.
#
# Required folder structure (all in same folder as this script):
#   preprocess.py
#   train_dataset_1/
#   train_dataset_2/
#   seen_subjects_test_set/
#   unseen_subjects_test_set/
#
# Shared settings (do not change):
#   Input channels : accelerometer x/y/z, gyroscope x/y/z, magnetometer x/y/z (9 total)
#   Window size    : 200 timesteps (1 second at 200Hz)
#   Stride         : 50 timesteps
#   Normalisation  : z-score per channel, fit on training set only
#   Output         : sin(heading) and cos(heading) — 2 values
#   Split          : train = train_dataset_1 + train_dataset_2
#                    val   = seen_subjects_test_set
#                    test  = unseen_subjects_test_set
# =============================================================


def create_windows(X, y, window_size=200, stride=50):
    X_windows = []
    y_windows = []
    
    for start in range(0, len(X) - window_size, stride):
        end = start + window_size
        X_windows.append(X[start:end])
        y_windows.append(y[end - 1])
    
    return np.array(X_windows), np.array(y_windows)


def load_sequence(seq_path):
    with h5py.File(seq_path, 'r') as f:
        acce = f['synced']['acce'][:]
        gyro = f['synced']['gyro'][:]
        magnet = f['synced']['magnet'][:]
        tango_pos = f['pose']['tango_pos'][:]
    
    dx = np.diff(tango_pos[:, 0])
    dy = np.diff(tango_pos[:, 1])
    heading = np.arctan2(dy, dx)
    
    acce = acce[1:]
    gyro = gyro[1:]
    magnet = magnet[1:]
    
    X = np.concatenate([acce, gyro, magnet], axis=1)
    y = np.column_stack([np.sin(heading), np.cos(heading)])
    
    return X, y


def load_dataset(folder):
    all_X = []
    all_y = []
    
    sequences = sorted(os.listdir(folder))
    
    for seq in sequences:
        seq_path = os.path.join(folder, seq, 'data.hdf5')
        if not os.path.exists(seq_path):
            continue
        X, y = load_sequence(seq_path)
        X_win, y_win = create_windows(X, y)
        all_X.append(X_win)
        all_y.append(y_win)
    
    return np.concatenate(all_X, axis=0), np.concatenate(all_y, axis=0)


def preprocess(train_folders, val_folder, test_folder):
    print("Loading training data...")
    X_train_list = []
    y_train_list = []
    for folder in train_folders:
        X, y = load_dataset(folder)
        X_train_list.append(X)
        y_train_list.append(y)
    X_train = np.concatenate(X_train_list, axis=0)
    y_train = np.concatenate(y_train_list, axis=0)
    
    print("Loading validation data...")
    X_val, y_val = load_dataset(val_folder)
    
    print("Loading test data...")
    X_test, y_test = load_dataset(test_folder)
    
    print("Normalising...")
    mean = X_train.mean(axis=(0, 1))
    std = X_train.std(axis=(0, 1))
    
    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std
    
    print("Saving arrays to disk...")
    np.save('X_train.npy', X_train)
    np.save('y_train.npy', y_train)
    np.save('X_val.npy', X_val)
    np.save('y_val.npy', y_val)
    np.save('X_test.npy', X_test)
    np.save('y_test.npy', y_test)
    np.save('mean.npy', mean)
    np.save('std.npy', std)
    
    print("Done. All arrays saved.")
    print("X_train:", X_train.shape, "y_train:", y_train.shape)
    print("X_val:", X_val.shape, "y_val:", y_val.shape)
    print("X_test:", X_test.shape, "y_test:", y_test.shape)
    
    return X_train, y_train, X_val, y_val, X_test, y_test, mean, std


# --- RUN ---
train_folders = ['train_dataset_1', 'train_dataset_2']
val_folder = 'seen_subjects_test_set'
test_folder = 'unseen_subjects_test_set'

X_train, y_train, X_val, y_val, X_test, y_test, mean, std = preprocess(
    train_folders, val_folder, test_folder
)


## Run the below snippet everytime you restart you jupyter notebook so that whatever pre processing you already did does not need to run again as in the abive script we already saved the pre processed files

In [ ]:
import numpy as np

X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_val = np.load('X_val.npy')
y_val = np.load('y_val.npy')
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

## As a contingency plan in case this script gives you trouble, just run the script that is present in the "testing pre-processing script" file (obviously without the run command codeblock)